# Step 3: Predict Fraud

**Goal**: Use trained model to detect fraud

- Load model
- Load test transactions
- Predict fraud probability
- Show results

In [0]:
%pip install xgboost
dbutils.library.restartPython()

In [0]:
# Load model
import pickle
from pyspark.sql.functions import hour, dayofweek, month
import pandas as pd

print("Loading model...")
model_row = spark.read.table("workspace.models.fraud_model").collect()[0]
model_artifact = pickle.loads(model_row['model_binary'])
model = model_artifact['model']

print(f"✓ Loaded: {model_row['model_name']}")
print(f"✓ Saved at: {model_row['saved_at']}")

In [0]:
# Load test transactions
print("Loading test data...")
df = spark.read.table("workspace.bronze.fraud_data").limit(100)

df_features = df.select(
    "txn_id",
    hour("timestamp").alias("hour"),
    dayofweek("timestamp").alias("day_of_week"),
    month("timestamp").alias("month"),
    "amount_inr",
    "is_fraud"
)

df_pandas = df_features.toPandas()
print(f"✓ Loaded {len(df_pandas)} transactions")

In [0]:
# Predict
X = df_pandas[['hour', 'day_of_week', 'month', 'amount_inr']]
predictions = model.predict(X)
predictions_proba = model.predict_proba(X)

df_pandas['predicted_fraud'] = predictions
df_pandas['fraud_prob'] = predictions_proba[:, 1]
df_pandas['prediction'] = df_pandas['predicted_fraud'].map({0: 'Legitimate', 1: 'FRAUD'})

print(f"✓ Predictions made")
print(f"\nFraud detected: {predictions.sum()}")
print(f"Accuracy: {(predictions == df_pandas['is_fraud']).mean():.2%}")

In [0]:
# Show results
print("\nTop 10 highest fraud probability:\n")
results = df_pandas[['txn_id', 'amount_inr', 'fraud_prob', 'prediction', 'is_fraud']]
results = results.sort_values('fraud_prob', ascending=False).head(10)
print(results.to_string(index=False))